In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# === Importation de vos modules ===
# Assurez-vous que ces fichiers sont dans le même dossier que le notebook
from market_env import MarketEnv
from ddrl_agent import DDRLAgent
from matrices import make_market_params

# === Configuration Graphique ===
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# === 1. Configuration et Chargement ===
device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = "checkpoint_multiasset.pt"
fallback_model_path = "policy_multiasset_quadratic.pt"

print(f"Device utilisé : {device}")

# Logique de chargement intelligente
try:
    if os.path.exists(checkpoint_path):
        print(f"Chargement du checkpoint complet : {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Cas 1 : Le fichier contient bien les paramètres de marché (Le cas idéal)
        if 'market_params' in checkpoint:
            print("✅ Paramètres de marché trouvés dans le checkpoint. Synchronisation Environnement...")
            mp = checkpoint['market_params']
            
            # Recréation de l'environnement EXACT de l'entraînement
            env = MarketEnv(
                alpha_weights=mp['alpha_weights'],
                omega=mp['omega'],
                return_weights=mp['return_weights'],
                sigma=mp['sigma'],
                trader_risk=mp['trader_risk'],
                dealer_risk=mp['dealer_risk'],
                horizon=mp['horizon'],
                device=device
            )
            
            agent = DDRLAgent(env=env, horizon=mp['horizon'], hidden_dim=300, device=device)
            agent.policy.load_state_dict(checkpoint['model_state_dict'])
            
        else:
            # Cas 2 : Checkpoint existe mais ancien format (juste les poids)
            print("⚠️ Checkpoint trouvé mais sans paramètres de marché. Utilisation de paramètres ALÉATOIRES (Performance dégradée attendue).")
            raise KeyError("market_params missing")
            
    elif os.path.exists(fallback_model_path):
        print(f"⚠️ Checkpoint complet introuvable. Chargement du modèle simple : {fallback_model_path}")
        print("⚠️ Utilisation de paramètres ALÉATOIRES (Performance dégradée attendue).")
        raise FileNotFoundError("Checkpoint missing")
        
    else:
        print("❌ Aucun modèle trouvé. Initialisation aléatoire totale.")
        raise FileNotFoundError("No model found")

except (KeyError, FileNotFoundError):
    # FALLBACK : Génération de nouveaux paramètres (ce qui causait votre problème de performance)
    # On le garde pour que le code ne plante pas si vous n'avez pas encore ré-entrainé.
    num_assets = 2
    num_alphas = 2
    horizon = 50
    trader_risk = 1e-6
    dealer_risk = 1e-6
    
    return_weights, alpha_weights, sigma, omega = make_market_params(num_assets, num_alphas, device)
    
    env = MarketEnv(
        alpha_weights=alpha_weights,
        omega=omega,
        return_weights=return_weights,
        sigma=sigma,
        trader_risk=trader_risk,
        dealer_risk=dealer_risk,
        horizon=horizon,
        device=device
    )
    
    agent = DDRLAgent(env=env, horizon=horizon, hidden_dim=300, device=device)
    
    # Tentative de chargement des poids seuls si disponibles
    if os.path.exists(fallback_model_path):
        agent.policy.load_state_dict(torch.load(fallback_model_path, map_location=device))

agent.policy.eval()

# === 2. Rollout Personnalisé (Extraction Historique) ===

def custom_rollout(agent, num_samples=1000):
    print(f"Génération de {num_samples} scénarios de test...")
    U_test = agent.env.generate_randomness(num_samples)
    
    # Stockage
    all_positions = []  # Actions (w)
    all_signals = []    # Signal composite (B * alpha)
    rewards = torch.zeros(num_samples, device=device)
    
    with torch.no_grad():
        state = agent.env.reset(num_samples)
        
        for t in range(agent.horizon):
            # Extraction
            alpha_t = state[:, 0:agent.env.num_alphas]
            
            # Action
            action = agent.policy(state) # w_t
            
            # Calcul du Signal Composite (Alpha @ B.T)
            # C'est la partie prédictible du rendement
            composite_signal = alpha_t @ agent.env.return_weights.T
            
            # Stockage
            all_positions.append(action.cpu())
            all_signals.append(composite_signal.cpu())
            
            # Step
            rewards += agent.env.reward(state, action)
            state = agent.env.transition(state, action, U_test[:, t, :])
            
    return {
        "positions": torch.stack(all_positions, dim=1).numpy(), # (Batch, T, Assets)
        "signals": torch.stack(all_signals, dim=1).numpy(),     # (Batch, T, Assets)
        "rewards": rewards.cpu().numpy()
    }

history = custom_rollout(agent, num_samples=2000)

# === 3. Visualisation 1 : Double Axe (Position vs Signal) ===
# CORRECTION : Utilisation de twinx() pour voir le signal même s'il est petit

sample_idx = 0  # Premier scénario
asset_idx = 0   # Premier actif

time_steps = np.arange(agent.horizon)
signal_data = history["signals"][sample_idx, :, asset_idx]
position_data = history["positions"][sample_idx, :, asset_idx]

fig, ax1 = plt.subplots(figsize=(12, 6))

# Axe Gauche : Position (Rouge)
color_pos = 'tab:red'
ax1.set_xlabel('Temps (Horizon)')
ax1.set_ylabel('Position Agent (w)', color=color_pos, fontsize=12, fontweight='bold')
ax1.plot(time_steps, position_data, color=color_pos, linewidth=2, label='Position (w)')
ax1.tick_params(axis='y', labelcolor=color_pos)
ax1.grid(True, alpha=0.3)

# Axe Droit : Signal (Bleu) - Échelle différente !
ax2 = ax1.twinx() 
color_sig = 'tab:blue'
ax2.set_ylabel('Signal Composite (Alpha * B)', color=color_sig, fontsize=12, fontweight='bold')
ax2.plot(time_steps, signal_data, color=color_sig, linestyle='--', linewidth=2, label='Signal Predictif')
ax2.tick_params(axis='y', labelcolor=color_sig)
ax2.grid(False) # On évite de surcharger la grille

plt.title(f"Dynamique Intra-Scénario : Actif {asset_idx} (Double Axe)", fontsize=14)
fig.tight_layout()
plt.show()

print("Note : L'axe de droite (Signal) a une échelle beaucoup plus petite que l'axe de gauche.")

# === 4. Visualisation 2 : Corrélation Globale ===
# On vérifie que l'agent achète bien quand le signal est positif

flat_signals = history["signals"][:, :, asset_idx].flatten()
flat_positions = history["positions"][:, :, asset_idx].flatten()

# Sous-échantillonnage pour la rapidité d'affichage
idx = np.random.choice(len(flat_signals), 5000, replace=False)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=flat_signals[idx], y=flat_positions[idx], alpha=0.15, color="purple", edgecolor=None)

# Régression linéaire pour visualiser la tendance
m, b = np.polyfit(flat_signals[idx], flat_positions[idx], 1)
plt.plot(flat_signals[idx], m*flat_signals[idx] + b, color='black', linestyle='--', linewidth=1.5, label=f"Tendance (Pente: {m:.2f})")

plt.title(f"Corrélation Signal vs Position (Actif {asset_idx}) - 5000 points aléatoires")
plt.xlabel("Valeur du Signal Composite")
plt.ylabel("Position prise par l'Agent")
plt.legend()
plt.show()

# === 5. Visualisation 3 : Eigen-Portfolios (Analyse PCA) ===
# L'agent a-t-il appris la structure de corrélation ?

# 1. Récupération de la matrice de covariance des rendements (Sigma)
cov_mat = agent.env.sigma.cpu().numpy()

# 2. Décomposition en valeurs/vecteurs propres
eigenvals, eigenvecs = np.linalg.eigh(cov_mat) 
# eigenvecs est (N_assets, N_assets), chaque colonne est un vecteur propre

# 3. Projection des positions moyennes de l'agent sur ces vecteurs
# On prend la moyenne des positions absolues pour voir l'exposition au risque "en magnitude"
mean_abs_position = np.abs(history["positions"]).mean(axis=(0,1)) # (Num_Assets,)

# Projection : P_eigen = V.T @ P_asset
# Cela nous dit combien l'agent investit dans le "Facteur 1", "Facteur 2", etc.
eigen_exposures = eigenvecs.T @ mean_abs_position

plt.figure(figsize=(8, 5))
bar_colors = sns.color_palette("viridis", len(eigen_exposures))
plt.bar(range(len(eigen_exposures)), eigen_exposures, color=bar_colors)

plt.title("Exposition Moyenne aux Eigen-Portfolios (Facteurs de Risque)")
plt.xlabel("Indice du Facteur (0 = Plus faible volatilité -> N = Plus forte)")
plt.ylabel("Magnitude Moyenne de l'Exposition")
plt.xticks(range(len(eigen_exposures)), [f"Facteur {i}" for i in range(len(eigen_exposures))])
plt.grid(axis='y', alpha=0.3)
plt.show()

# === 6. Distribution des Profits ===
mean_reward = history["rewards"].mean()
plt.figure(figsize=(10, 5))
sns.histplot(history["rewards"], kde=True, color="green", bins=60, alpha=0.6)
plt.axvline(mean_reward, color='red', linestyle='--', linewidth=2, label=f'Moyenne: {mean_reward:.2f}')
plt.title(f"Distribution du P&L (Moyenne sur {history['rewards'].shape[0]} scénarios)")
plt.xlabel("Cumulative Reward")
plt.legend()
plt.show()
